In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
MONOTONE (NEVER-INCREASE) BW IMPROVEMENT FROM AN INITIAL RANKING
===============================================================

You asked for: "improve it in any ways possible" while guaranteeing that the
sum of backward edges (BW) NEVER increases.

Your adjacent-swap version can get stuck immediately (0 swaps), so this
version upgrades the move set to WINDOW MOVES of size k:

Move type (window insertion):
  Pick a node x at position i and move it to position j within [i-k, i+k].
  This is equivalent to a sequence of adjacent swaps, but we evaluate the
  whole move's delta exactly and apply the best improving move.

BW delta for moving x across one neighbor y:
  If x crosses y from left->right (x was before y, becomes after):
      delta += w(x->y) - w(y->x)
  If x crosses y from right->left (x was after y, becomes before):
      delta += w(y->x) - w(x->y)

So an insertion move's delta is the sum of those pairwise crossing deltas
over the crossed segment. This is computed in O(k) time with O(1) edge lookups.

Guarantee:
  - We only accept moves with delta_BW < 0 (strictly improving BW), so BW is
    monotonically non-increasing, step by step.
  - Optional: allow delta_BW == 0 "plateau moves" ONLY if they strictly improve
    a secondary potential (sum of abs(position - initial_position) over the moved
    window). This still guarantees BW never increases, and cannot cycle.

This is designed to actually change something on big graphs like connectome.

Inputs:
  - dimacs_path: .d graph
  - init_ranking_csv: CSV with columns (Node ID, Order)
Outputs:
  - out_csv: improved ranking CSV (Node ID, Order)
"""

import time
import pandas as pd
from collections import defaultdict


# ---------------------------
# DIMACS reader (aggregating parallel arcs)
# ---------------------------

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ---------------------------
# Objective computation
# ---------------------------

def compute_forward_backward(edges_indexed, pos):
    total_w = 0.0
    fw = 0.0
    for u, v, w in edges_indexed:
        total_w += w
        if pos[u] < pos[v]:
            fw += w
    bw = total_w - fw
    return total_w, fw, bw


# ---------------------------
# Initial ranking loader
# ---------------------------

def load_initial_order_csv(init_ranking_csv_path, node_to_index):
    df = pd.read_csv(init_ranking_csv_path)
    if "Node ID" not in df.columns or "Order" not in df.columns:
        raise ValueError("Initial ranking CSV must have columns: 'Node ID' and 'Order'")

    pairs = []
    seen = set()
    for _, row in df.iterrows():
        nid = str(row["Node ID"]).strip()
        if nid not in node_to_index:
            continue
        idx = node_to_index[nid]
        if idx in seen:
            continue
        seen.add(idx)
        try:
            ordv = int(row["Order"])
        except Exception:
            ordv = 0
        pairs.append((ordv, idx))

    pairs.sort(key=lambda t: (t[0], t[1]))
    order = [idx for _, idx in pairs]

    # append missing nodes deterministically
    n = len(node_to_index)
    if len(order) < n:
        for i in range(n):
            if i not in seen:
                order.append(i)

    if len(order) != n:
        raise RuntimeError("Failed to construct a full initial order.")
    return order


# ---------------------------
# Fast edge lookup: out_w[u].get(v, 0.0)
# ---------------------------

def build_out_weight_maps(edges_indexed, n):
    out_w = [dict() for _ in range(n)]
    for u, v, w in edges_indexed:
        out_w[u][v] = w
    return out_w


# ---------------------------
# Window-move monotone improvement
# ---------------------------

def _delta_move_insertion(order, out_w, i, j):
    """
    Exact delta_BW for moving node x at position i to position j (insertion),
    where i != j.

    Uses the pairwise crossing formula; O(|i-j|).
    """
    if i == j:
        return 0.0

    x = order[i]
    delta = 0.0

    if j > i:
        # x moves right, crosses nodes y in (i+1..j)
        # each crossing contributes: w(x->y) - w(y->x)
        for t in range(i + 1, j + 1):
            y = order[t]
            delta += out_w[x].get(y, 0.0) - out_w[y].get(x, 0.0)
    else:
        # x moves left, crosses nodes y in (j..i-1)
        # each crossing contributes: w(y->x) - w(x->y)
        for t in range(j, i):
            y = order[t]
            delta += out_w[y].get(x, 0.0) - out_w[x].get(y, 0.0)

    return delta


def _secondary_potential_local(order, pos0, L, R):
    """
    Secondary potential over a local window [L, R] inclusive:
      sum_{t=L..R} abs(t - pos0[order[t]])
    Used only to accept plateau moves (delta_BW == 0) without cycling.
    """
    s = 0
    for t in range(L, R + 1):
        u = order[t]
        s += abs(t - pos0[u])
    return s


def improve_ranking_monotone_bw_window(
    edges_indexed,
    init_order,
    *,
    k=50,
    time_limit_s=600.0,
    max_passes=50,
    eps=1e-12,
    allow_plateau_moves=True,
    plateau_budget=2000,
    verbose_every_pass=1,
):
    """
    Window insertion local search with strict-improvement BW guarantee.

    - For each index i, try moving order[i] to j in [i-k, i+k].
    - Choose the best (most negative delta) move in that neighborhood and apply it.
    - BW never increases because we accept only delta < 0 (or delta==0 only if
      it improves a secondary potential and plateau budget allows).

    Returns:
      order_best, bw0, bw_best, info
    """
    n = len(init_order)
    order = list(init_order)
    out_w = build_out_weight_maps(edges_indexed, n)

    # initial positions
    pos = [0] * n
    for t, u in enumerate(order):
        pos[u] = t

    # pos0 for plateau acceptance (secondary potential)
    pos0 = list(pos)

    total_w, fw, bw = compute_forward_backward(edges_indexed, pos)
    bw0 = bw

    t0 = time.perf_counter()
    passes = 0
    moves = 0
    plateau_used = 0

    while passes < max_passes:
        if time_limit_s is not None and (time.perf_counter() - t0) >= time_limit_s:
            break

        improved_this_pass = 0
        plateau_this_pass = 0

        # sweep i from left to right
        i = 0
        while i < n:
            if time_limit_s is not None and (time.perf_counter() - t0) >= time_limit_s:
                break

            L = max(0, i - k)
            R = min(n - 1, i + k)

            best_j = i
            best_delta = 0.0
            best_is_plateau = False
            best_sec_gain = 0  # positive means secondary decreases

            # Evaluate all candidate targets in window
            # (You can prune by only checking a subset if needed, but full window is strongest.)
            for j in range(L, R + 1):
                if j == i:
                    continue

                delta = _delta_move_insertion(order, out_w, i, j)

                if delta < best_delta - eps:
                    best_delta = delta
                    best_j = j
                    best_is_plateau = False
                    best_sec_gain = 0

                elif allow_plateau_moves and abs(delta) <= eps and plateau_used < plateau_budget:
                    # Plateau move (delta == 0): accept only if it improves secondary potential locally
                    # Evaluate secondary only for a small local range affected by move
                    a = min(i, j)
                    b = max(i, j)
                    sec_before = _secondary_potential_local(order, pos0, a, b)

                    # simulate insertion locally (cheap because window size <= k)
                    x = order[i]
                    if j > i:
                        new_segment = order[a:i] + order[i+1:j+1] + [x] + order[j+1:b+1]
                    else:
                        new_segment = order[a:j] + [x] + order[j:i] + order[i+1:b+1]

                    # compute sec_after for that segment
                    sec_after = 0
                    for idx_local, u in enumerate(new_segment, start=a):
                        sec_after += abs(idx_local - pos0[u])

                    if sec_after < sec_before:
                        # secondary improved; take it if no negative delta move found
                        # We prefer negative delta always; plateau only if best_delta is still 0.
                        if best_delta > -eps:
                            best_delta = 0.0
                            best_j = j
                            best_is_plateau = True
                            best_sec_gain = (sec_before - sec_after)

            # Apply best move if it improves BW, or plateau-improves secondary
            if best_j != i and (best_delta < -eps or best_is_plateau):
                x = order[i]
                # Remove x
                order.pop(i)
                # Insert at new position (note: after pop, indices shift if j>i)
                j_ins = best_j
                if best_j > i:
                    j_ins = best_j  # because list is shorter by 1 before insertion at the right end
                order.insert(j_ins, x)

                # Update positions for affected range only
                a = min(i, j_ins)
                b = max(i, j_ins)
                for t in range(a, b + 1):
                    pos[order[t]] = t

                # BW update: exact for negative moves; for plateau moves it's unchanged
                if best_delta < -eps:
                    bw += best_delta
                    improved_this_pass += 1
                else:
                    plateau_used += 1
                    plateau_this_pass += 1

                moves += 1

                # after moving, it's often good to step back a bit
                i = max(0, a - 1)
                continue

            i += 1

        passes += 1

        if verbose_every_pass and (passes % verbose_every_pass == 0):
            elapsed = time.perf_counter() - t0
            print(f"[pass {passes}] BW={bw:.6f}  improving_moves={improved_this_pass}  plateau_moves={plateau_this_pass}  elapsed_s={elapsed:.2f}")

        if improved_this_pass == 0 and (not allow_plateau_moves or plateau_this_pass == 0):
            break

    # Sanity recompute BW
    _tot2, _fw2, bw_check = compute_forward_backward(edges_indexed, pos)
    if abs(bw_check - bw) > 1e-6:
        bw = bw_check

    # Hard guarantee vs initial
    if bw > bw0 + 1e-9:
        # Should not happen; return initial if it does
        return list(init_order), bw0, bw0, {
            "passes": 0, "moves": 0, "plateau_used": 0, "guard": True,
            "elapsed_s": time.perf_counter() - t0
        }

    return order, bw0, bw, {
        "passes": passes, "moves": moves, "plateau_used": plateau_used, "guard": False,
        "elapsed_s": time.perf_counter() - t0
    }


# ---------------------------
# Write ranking CSV
# ---------------------------

def write_ranking_csv_from_order(order, index_to_node, out_csv_path):
    rows = [{"Node ID": str(index_to_node[u]).strip(), "Order": int(i)} for i, u in enumerate(order)]
    pd.DataFrame(rows).to_csv(out_csv_path, index=False)


# ---------------------------
# Main
# ---------------------------

if __name__ == "__main__":
    # === EDIT THESE PATHS ===
    dimacs_path = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"
    init_ranking_csv = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/35463823.csv"  # must have columns Node ID, Order

    out_csv = dimacs_path.replace(".d", "") + "_monotone_bw_window_improved.csv"

    # Search parameters
    time_limit_s = 600.0
    k = 200                 # window radius; try 50, 200, 500
    max_passes = 25
    allow_plateau_moves = True
    plateau_budget = 5000   # number of 0-delta moves allowed total (still BW-safe)

    t_all = time.perf_counter()

    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(dimacs_path)
    n = len(node_to_index)
    m = len(edges_indexed)

    init_order = load_initial_order_csv(init_ranking_csv, node_to_index)

    improved_order, bw0, bw1, info = improve_ranking_monotone_bw_window(
        edges_indexed,
        init_order,
        k=k,
        time_limit_s=time_limit_s,
        max_passes=max_passes,
        eps=1e-12,
        allow_plateau_moves=allow_plateau_moves,
        plateau_budget=plateau_budget,
        verbose_every_pass=1,
    )

    write_ranking_csv_from_order(improved_order, index_to_node, out_csv)

    # Final stats
    pos = [0] * n
    for i, u in enumerate(improved_order):
        pos[u] = i
    total_w, fw, bw = compute_forward_backward(edges_indexed, pos)

    elapsed = time.perf_counter() - t_all

    print("\n================= RESULT =================")
    print(f"✅ Wrote improved ranking: {out_csv}")
    print(f"Graph: n={n} m={m} (after aggregation)")
    print(f"Total Weight:   {total_w:.6f}")
    print(f"Forward Weight: {fw:.6f}")
    print(f"Backward Weight:{bw:.6f}")
    print(f"Forward Ratio:  {fw/total_w if total_w>0 else 0.0:.9f}")
    print(f"BW initial:     {bw0:.6f}")
    print(f"BW improved:    {bw1:.6f}   (GUARANTEED <= initial)")
    print(f"Passes: {info['passes']}  Moves: {info['moves']}  PlateauUsed: {info['plateau_used']}  GuardTriggered: {info['guard']}")
    print(f"⏱️ Total runtime: {elapsed:.3f} s   Search time: {info['elapsed_s']:.3f} s")


[pass 1] BW=6448312.000000  improving_moves=4  plateau_moves=18  elapsed_s=600.00

================= RESULT =================
✅ Wrote improved ranking: /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome_monotone_bw_window_improved.csv
Graph: n=136648 m=5657719 (after aggregation)
Total Weight:   41912141.000000
Forward Weight: 35463829.000000
Backward Weight:6448312.000000
Forward Ratio:  0.846146920
BW initial:     6448318.000000
BW improved:    6448312.000000   (GUARANTEED <= initial)
Passes: 1  Moves: 22  PlateauUsed: 18  GuardTriggered: False
⏱️ Total runtime: 614.577 s   Search time: 600.633 s
